# Agent Diagnosis

This notebook classifies **unique symptom combinations** (ProEpi/Guardiões da Saúde, PT-BR) against the **Medley Disease and symptoms dataset 2023** (EN) using **Jaccard similarity** and two diagnosis tools.

## How the agent works

- The agent **reads each row** of the dataset **agent_diagnosis_sintomas_unicos** and, for each row, **searches** the **Medley Disease and symptoms dataset 2023** to assign diagnoses.
- **Two tools (Jaccard-based):**
  - **Tool 1 – Primary diagnosis:** Always the best-matching disease (highest Jaccard); if none ≥ 60%, still uses that best match.
  - **Tool 2 – Secondary diagnoses:** Diseases with 50–59% Jaccard **only when** primary has ≥ 60%; otherwise empty.
- **Helper:** PT-BR symptom names are mapped to the English column names of the Medley Disease and symptoms dataset 2023 before computing Jaccard. Tie-breaking by disease frequency in that dataset (higher first).
- **Reference dataset:** Medley Disease and symptoms dataset 2023 — 773 diseases, 377 symptoms, ~246k rows.

## 1. Imports and path configuration

- Import libraries (os, pandas, numpy, pathlib, collections).
- Define absolute paths: ProEpi (input), Medley Disease and symptoms dataset 2023 (reference), output file.
- Create output directory if it does not exist.

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

PATH_PROEPI = "/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/results/clusters_outputs/clusters_outputs_dataset_sintomas_grupos.xlsx"
PATH_MEDLEY = "/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/inputs/Disease and symptoms dataset.csv"
PATH_OUTPUT = "/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/results/agent_outputs/agent_outputs_dataset_sintomas_grupos_classificado.xlsx"
PATH_OUTPUT_UNICOS = "/Users/daniellybx/Documents/Projeto ProEpi GdS /projeto_proepi_gds_datascience/data/results/agent_outputs/agent_outputs_dataset_sintomas_agrupados_unicos_classificado.xlsx"

os.makedirs(os.path.dirname(PATH_OUTPUT), exist_ok=True)
print("Paths configured.")

## 2. ProEpi data handling and unique symptoms dataset

- Load ProEpi Excel (clusters_outputs_dataset_sintomas_grupos.xlsx).
- Identify the 42 symptom columns (positions 7 to 48).
- Deduplicate by these columns to keep only unique combinations.
- Build dataframe agent_diagnosis_sintomas_unicos with unique_id, symptom columns, primary_diagnosis and secondary_diagnoses (initially empty).

In [ ]:
df_proepi = pd.read_excel(PATH_PROEPI, engine="openpyxl")
cols_sintomas = df_proepi.columns[6:48].tolist()

agent_diagnosis_sintomas_unicos = df_proepi[cols_sintomas].drop_duplicates().reset_index(drop=True)
agent_diagnosis_sintomas_unicos.insert(0, "unique_id", np.arange(len(agent_diagnosis_sintomas_unicos)))
agent_diagnosis_sintomas_unicos["primary_diagnosis"] = ""
agent_diagnosis_sintomas_unicos["secondary_diagnoses"] = ""

print(f"ProEpi: {len(df_proepi)} rows. Unique symptom sets: {len(agent_diagnosis_sintomas_unicos)}.")
agent_diagnosis_sintomas_unicos.head()

## 3. Medley Disease and symptoms dataset 2023 — loading and pre-aggregation by disease

- Load the Medley Disease and symptoms dataset 2023 CSV (diseases and symptoms as dummy columns).
- List of symptom columns (all except "diseases").
- Disease frequency dictionary (row counts per disease).
- Function that groups by disease and returns the set of present symptoms (columns with value 1) per disease.

In [ ]:
medley = pd.read_csv(PATH_MEDLEY)
col_disease = "diseases"
cols_medley_sintomas = [c for c in medley.columns if c != col_disease]

doenca_frequencia = medley[col_disease].value_counts().to_dict()

def sintomas_por_doenca(medley_df, col_doenca, cols_sint):
    out = {}
    for doenca, grp in medley_df.groupby(col_doenca):
        row = grp[cols_sint].max(axis=0)
        out[doenca] = set(row[row > 0].index.tolist())
    return out

doenca_sintomas = sintomas_por_doenca(medley, col_disease, cols_medley_sintomas)
print(f"Medley Disease and symptoms dataset 2023: {len(medley)} rows, {len(doenca_sintomas)} diseases, {len(cols_medley_sintomas)} symptoms.")

## 4. PT-BR to EN mapping (semantic translation)

- Dictionary mapping each of the 42 ProEpi symptom columns (PT-BR) to the corresponding column name in the Medley Disease and symptoms dataset 2023 (EN).
- Function that, given a row of ProEpi dummies, returns the set of present (value 1) and mapped symptom names in English.

In [ ]:
MAPEAMENTO_PT_EN = {
    "Bolhasna Pele": "skin lesion",
    "Calafrios": "chills",
    "Cansaco": "fatigue",
    "Coceira": "itching of skin",
    "Coloraçãoazuladanoslábiosourosto": "pallor",
    "CongestãoNasal": "nasal congestion",
    "Conjuntivite": "eye redness",
    "Coriza": "coryza",
    "Diarreia": "diarrhea",
    "DificuldadeParaRespirar": "difficulty breathing",
    "Diminuiçãodaurina": "low urine output",
    "Diminuiçãodoapetite": "decreased appetite",
    "DorCabeca": "headache",
    "DorDeEstômago": "sharp abdominal pain",
    "DorNasArticulações": "joint pain",
    "DorNosMúsculos": "muscle pain",
    "DorNosOlhos": "pain in eye",
    "Doratrásdosolhos": "eye strain",
    "DordeGarganta": "sore throat",
    "DornoPeito": "sharp chest pain",
    "Dornocorpo": "ache all over",
    "Dornojoelho": "knee pain",
    "Exantema(manchasavermelhadasnocorpo)": "skin rash",
    "Fadiga": "fatigue",
    "Febre": "fever",
    "Inchaçonaspálpebras": "eyelid swelling",
    "InguaOuGângliosInflamados": "swollen lymph nodes",
    "Irritabilidade": "restlessness",
    "Mal-estar": "feeling ill",
    "ManchasRoxasPeloCorpo": "skin lesion",
    "NáuseaOuVômito": "nausea",
    "Olhosgrudadosaoacordar": "white discharge from eye",
    "Olhosvermelhoselacrimejantes": "lacrimation",
    "PeleEOlhosAmarelados": "jaundice",
    "Sangramento": "bleeding gums",
    "Sangueoumuconasfezes": "blood in stool",
    "Secreçãoesbranquiçadaouamarelada": "vaginal discharge",
    "Sensaçãodeareianosolhos": "foreign body sensation in eye",
    "Sensibilidadeàluz": "eye strain",
    "Tosse": "cough",
    "Visãoembaçada": "diminished vision",
    "paladareolfato": "disturbance of smell or taste",
}

def sintomas_pt_to_en(row_sintomas, cols_sintomas_pt, mapeamento):
    en_set = set()
    for col in cols_sintomas_pt:
        if col in row_sintomas.index and row_sintomas[col] and int(row_sintomas[col]) == 1:
            if col in mapeamento and mapeamento[col] in cols_medley_sintomas:
                en_set.add(mapeamento[col])
    return en_set

print(f"Mapping: {len(MAPEAMENTO_PT_EN)} PT-BR columns -> EN.")

## 5. Jaccard similarity and tie-breaking functions

- **Jaccard similarity:** |A ∩ B| / |A ∪ B| between the patient's symptom set (mapped to EN) and each disease's symptom set in the Medley Disease and symptoms dataset 2023. Used for both primary and secondary diagnosis tools.
- **Primary:** best match among all diseases; if none ≥ 60%, still use the highest Jaccard. **Secondary:** 50% ≤ Jaccard < 60%, only when primary has Jaccard ≥ 60%. Order by Jaccard (descending), then disease frequency.

In [ ]:
def jaccard_similarity(sintomas_paciente_en, sintomas_doenca):
    if not sintomas_paciente_en and not sintomas_doenca:
        return 0.0
    inter = len(sintomas_paciente_en & sintomas_doenca)
    uniao = len(sintomas_paciente_en | sintomas_doenca)
    return inter / uniao if uniao else 0.0

def ranquear_doencas(sintomas_paciente_en, doenca_sintomas_dict, doenca_freq_dict, limiar_min):
    candidatos = []
    for doenca, sym_set in doenca_sintomas_dict.items():
        jacc = jaccard_similarity(sintomas_paciente_en, sym_set)
        if jacc >= limiar_min:
            candidatos.append((doenca, jacc, doenca_freq_dict.get(doenca, 0)))
    candidatos.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return [(d, p) for d, p, _ in candidatos]

def ranquear_doencas_intervalo(sintomas_paciente_en, doenca_sintomas_dict, doenca_freq_dict, limiar_min, limiar_max):
    candidatos = []
    for doenca, sym_set in doenca_sintomas_dict.items():
        jacc = jaccard_similarity(sintomas_paciente_en, sym_set)
        if limiar_min <= jacc < limiar_max:
            candidatos.append((doenca, jacc, doenca_freq_dict.get(doenca, 0)))
    candidatos.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return [(d, p) for d, p, _ in candidatos]

def ranquear_todas_doencas(sintomas_paciente_en, doenca_sintomas_dict, doenca_freq_dict):
    candidatos = []
    for doenca, sym_set in doenca_sintomas_dict.items():
        jacc = jaccard_similarity(sintomas_paciente_en, sym_set)
        candidatos.append((doenca, jacc, doenca_freq_dict.get(doenca, 0)))
    candidatos.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return [(d, jacc) for d, jacc, _ in candidatos]

## 6. DiagnosisAgent class and all tools

- **Helper – Semantic translation:** converts a PT-BR symptom row into the set of English symptom names used in the Medley Disease and symptoms dataset 2023.
- **Tool 1 – Primary diagnosis:** Always returns the **best-matching disease** (highest Jaccard, tie-break by frequency). If no disease reaches 60%, still returns that best match (no "Insufficient symptoms").
- **Tool 2 – Secondary diagnoses:** Returns diseases with 50% ≤ Jaccard < 60% **only when** the primary diagnosis has Jaccard ≥ 60%. If primary is below 60%, returns an empty list.
- **Tool 3 – Translate primary:** translates the primary diagnosis name from English to PT-BR using LLaMA (Ollama); cache per unique string.
- **Tool 4 – Translate secondaries:** translates the semicolon-separated secondary diagnoses from English to PT-BR using LLaMA (Ollama); cache per unique string.
- The pipeline uses these tools so that the columns primary_diagnosis and secondary_diagnoses are filled **already in PT-BR** (see section 7).

In [ ]:
import os
import subprocess
import time
os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")

import ollama

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
_translation_cache = {}
_ollama_client = ollama.Client(host=OLLAMA_HOST)

def _ensure_ollama_running():
    for attempt in range(2):
        try:
            _ollama_client.list()
            if attempt > 0:
                print("Ollama is now running.")
            return True
        except Exception as e:
            if attempt == 0:
                print("Ollama not running. Starting 'ollama serve' in the background...")
                try:
                    subprocess.Popen(
                        ["ollama", "serve"],
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL,
                        start_new_session=True,
                    )
                except FileNotFoundError:
                    print("  'ollama' not found in PATH. Install Ollama from https://ollama.com and add it to PATH.")
                    return False
                for _ in range(12):
                    time.sleep(1)
                    try:
                        _ollama_client.list()
                        print("Ollama started successfully.")
                        return True
                    except Exception:
                        pass
                print("  Ollama did not start in time. Try running 'ollama serve' in a terminal.")
                return False
            raise e
    return False

if not _ensure_ollama_running():
    raise RuntimeError("Ollama could not be started. Install from https://ollama.com and run 'ollama serve' in a terminal.")

def _pick_ollama_model():
    preferred = ["llama3.2", "llama3.1", "llama3", "mistral", "phi3", "phi", "llama2", "gemma"]
    try:
        resp = _ollama_client.list()
        models = resp.get("models") or []
        names = [m.get("name") or m.get("model", "") for m in models if m.get("name") or m.get("model")]
        if not names:
            return None
        for p in preferred:
            for n in names:
                if p in n.lower():
                    return n
        return names[0]
    except Exception:
        return None

try:
    OLLAMA_MODEL = _pick_ollama_model()
    if OLLAMA_MODEL:
        print(f"Ollama connected at {OLLAMA_HOST}. Model for translation: {OLLAMA_MODEL}")
    else:
        OLLAMA_MODEL = "llama3.2"
        print(f"Ollama connected at {OLLAMA_HOST}. No model found — run: ollama pull {OLLAMA_MODEL} (or llama3, mistral, etc.)")
except Exception as e:
    OLLAMA_MODEL = "llama3.2"
    print(f"Ollama not reachable at {OLLAMA_HOST}. Start Ollama (open the app or run 'ollama serve'). Error: {e}")

def _translate_with_llama(en_text):
    if not en_text or str(en_text).strip() == "":
        return ""
    s = str(en_text).strip()
    if s in _translation_cache:
        return _translation_cache[s]
    prompt = (
        "Translate the following medical or disease name from English to Brazilian Portuguese. "
        "Reply with only the translation, nothing else. Use standard medical terminology in Portuguese. "
        "If it is a proper noun (e.g. COVID-19) or already in Portuguese, return it unchanged.\n\n"
        f"{s}"
    )
    try:
        response = _ollama_client.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
        raw = (response.get("message") or {}).get("content", "")
        out = raw.strip().split("\n")[0].strip() if raw else s
        if not out:
            out = s
    except Exception as e:
        print(f"  [Ollama] Translation failed for '{s[:50]}'. Start Ollama and run: ollama pull {OLLAMA_MODEL}. Error: {e}")
        out = s
    _translation_cache[s] = out
    return out

def tool_traduzir_diagnostico_primario(en_name):
    if pd.isna(en_name) or str(en_name).strip() == "":
        return ""
    return _translate_with_llama(en_name)

def tool_traduzir_diagnosticos_secundarios(en_str, sep="; "):
    if pd.isna(en_str) or str(en_str).strip() == "":
        return ""
    parts = [p.strip() for p in str(en_str).split(sep) if p.strip()]
    translated = [_translate_with_llama(p) for p in parts]
    return sep.join(translated)

class DiagnosisAgent:
    def __init__(self, doenca_sintomas, doenca_frequencia, cols_medley, mapeamento_pt_en, limiar_principal=0.60, limiar_secundario_min=0.50, limiar_secundario_max=0.60):
        self.doenca_sintomas = doenca_sintomas
        self.doenca_frequencia = doenca_frequencia
        self.cols_medley = cols_medley
        self.mapeamento_pt_en = mapeamento_pt_en
        self.limiar_principal = limiar_principal
        self.limiar_secundario_min = limiar_secundario_min
        self.limiar_secundario_max = limiar_secundario_max

    def tool_traduzir_sintomas(self, row_sintomas, cols_sintomas_pt):
        return sintomas_pt_to_en(row_sintomas, cols_sintomas_pt, self.mapeamento_pt_en)

    def tool_diagnostico_principal(self, sintomas_en):
        if not sintomas_en:
            return "Insufficient symptoms"
        ranking = ranquear_todas_doencas(sintomas_en, self.doenca_sintomas, self.doenca_frequencia)
        if not ranking:
            return "Insufficient symptoms"
        return ranking[0][0]

    def tool_diagnosticos_secundarios(self, sintomas_en):
        if not sintomas_en:
            return []
        ranking_todas = ranquear_todas_doencas(sintomas_en, self.doenca_sintomas, self.doenca_frequencia)
        if not ranking_todas or ranking_todas[0][1] < self.limiar_principal:
            return []
        return [d for d, _ in ranquear_doencas_intervalo(sintomas_en, self.doenca_sintomas, self.doenca_frequencia, self.limiar_secundario_min, self.limiar_secundario_max)]

agent = DiagnosisAgent(doenca_sintomas, doenca_frequencia, cols_medley_sintomas, MAPEAMENTO_PT_EN)
print("DiagnosisAgent and all tools (including LLaMA translation) ready.")

## 7. System prompt

- Instructs the agent to read **each row** of the dataset `agent_diagnosis_sintomas_unicos`, run a search in the **Medley Disease and symptoms dataset 2023**, and use **Tools 1–2** (primary and secondary diagnosis) then **Tools 3–4** (translation) so that the columns **primary_diagnosis** and **secondary_diagnoses** are filled **already translated to PT-BR**.

In [ ]:
SYSTEM_PROMPT = """You are a diagnosis agent. Your task is to process the dataset **agent_diagnosis_sintomas_unicos** row by row.

**Available tools (you must use all four):**
- **Tool 1 – Primary diagnosis:** Input: symptom set (EN). Output: best-matching disease name in English (highest Jaccard; tie-break by frequency).
- **Tool 2 – Secondary diagnoses:** Input: symptom set (EN). Output: list of disease names in English with 50–59% Jaccard (only when primary ≥ 60%); otherwise empty list.
- **Tool 3 – Translation (primary):** Input: one disease name in English. Output: the same name **translated to Brazilian Portuguese (PT-BR)** via LLaMA. Use this to convert the result of Tool 1 before writing to primary_diagnosis.
- **Tool 4 – Translation (secondaries):** Input: semicolon-separated list of disease names in English. Output: the same list **translated to PT-BR** via LLaMA. Use this to convert the result of Tool 2 before writing to secondary_diagnoses.

**Procedure for each row:**
1. Map the row's symptom columns (PT-BR) to English symptom names used in the Medley Disease and symptoms dataset 2023.
2. Call **Tool 1** with that symptom set → obtain primary disease name (EN). Then call **Tool 3** with that name → obtain primary in PT-BR. Write the PT-BR result into the column **primary_diagnosis**.
3. Call **Tool 2** with the same symptom set → obtain list of secondary diseases (EN). Then call **Tool 4** with that list (e.g. joined by "; ") → obtain secondaries in PT-BR. Write the PT-BR result into the column **secondary_diagnoses** (semicolon-separated).
4. **Mandatory:** The columns primary_diagnosis and secondary_diagnoses must contain **only Portuguese (PT-BR)** text. Never write English diagnosis names into these columns; always pass them through Tool 3 or Tool 4 first.

**Reference:** Medley Disease and symptoms dataset 2023 — 773 diseases, 377 symptoms, ~246k rows. Jaccard similarity is computed between the row's symptom set and each disease's symptom set."""
print("System prompt defined.")

## 8. Test run (5 rows)

- Classify the first 5 unique combinations with the agent.
- Print progress per row (unique_id, primary diagnosis, number of secondaries).
- Display the test dataframe at the end.

In [ ]:
N_TESTE = 5
df_teste = agent_diagnosis_sintomas_unicos.head(N_TESTE).copy()

_test_pt = tool_traduzir_diagnostico_primario("Insufficient symptoms")
print(f"Translation check: 'Insufficient symptoms' -> '{_test_pt}' (output must be PT-BR)")
print("Filling primary_diagnosis and secondary_diagnoses with PT-BR (Tools 3–4).\n")

for i in range(len(df_teste)):
    row = df_teste.iloc[i]
    sintomas_en = agent.tool_traduzir_sintomas(row, cols_sintomas)
    prin_en = agent.tool_diagnostico_principal(sintomas_en)
    sec_en = agent.tool_diagnosticos_secundarios(sintomas_en)
    prin_pt = tool_traduzir_diagnostico_primario(prin_en)
    sec_pt = tool_traduzir_diagnosticos_secundarios("; ".join(sec_en)) if sec_en else ""
    df_teste.iloc[i, df_teste.columns.get_loc("primary_diagnosis")] = prin_pt
    df_teste.iloc[i, df_teste.columns.get_loc("secondary_diagnoses")] = sec_pt
    print(f"  [{i+1}/{N_TESTE}] unique_id={row['unique_id']} -> primary: {prin_pt} | secondaries: {len(sec_en)}")

print("--- Test run completed (diagnoses in PT-BR) ---")
df_teste

## 9. Batch processing

- Classify all rows of agent_diagnosis_sintomas_unicos.
- Fill primary_diagnosis and secondary_diagnoses columns.
- Print progress every 500 rows.

In [ ]:
total = len(agent_diagnosis_sintomas_unicos)
print(f"Classifying {total} unique combinations (diagnoses output in PT-BR)...")

for i in range(total):
    row = agent_diagnosis_sintomas_unicos.iloc[i]
    sintomas_en = agent.tool_traduzir_sintomas(row, cols_sintomas)
    prin_en = agent.tool_diagnostico_principal(sintomas_en)
    sec_en = agent.tool_diagnosticos_secundarios(sintomas_en)
    prin_pt = tool_traduzir_diagnostico_primario(prin_en)
    sec_pt = tool_traduzir_diagnosticos_secundarios("; ".join(sec_en)) if sec_en else ""
    agent_diagnosis_sintomas_unicos.iloc[i, agent_diagnosis_sintomas_unicos.columns.get_loc("primary_diagnosis")] = prin_pt
    agent_diagnosis_sintomas_unicos.iloc[i, agent_diagnosis_sintomas_unicos.columns.get_loc("secondary_diagnoses")] = sec_pt
    if (i + 1) % 500 == 0 or i == 0:
        print(f"  {i+1}/{total}")

print("Batch completed.")
agent_diagnosis_sintomas_unicos.head(10)

## 10. Merge with original dataset

- Join the original ProEpi dataset with the classified unique-symptoms dataset using the 42 symptom columns as key.
- All original rows get primary_diagnosis and secondary_diagnoses filled.
- Drop any duplicate columns produced by the merge.

In [ ]:
df_classificado = df_proepi.merge(
    agent_diagnosis_sintomas_unicos,
    on=cols_sintomas,
    how="left",
    suffixes=("", "_dup")
)
cols_drop = [c for c in df_classificado.columns if c.endswith("_dup")]
df_classificado = df_classificado.drop(columns=cols_drop, errors="ignore")
print(f"Final dataset: {len(df_classificado)} rows. primary_diagnosis and secondary_diagnoses filled.")
df_classificado.head()

## 11. Save results

- Export the classified dataset (merge with original) to `PATH_OUTPUT` (agent_outputs_dataset_sintomas_grupos_classificado.xlsx).
- Export the unique symptom combinations with diagnoses to `PATH_OUTPUT_UNICOS` (agent_outputs_dataset_sintomas_agrupados_unicos_classificado.xlsx). Both in Excel (.xlsx) format.

In [ ]:
df_classificado.to_excel(PATH_OUTPUT, index=False, engine="openpyxl")
agent_diagnosis_sintomas_unicos.to_excel(PATH_OUTPUT_UNICOS, index=False, engine="openpyxl")
print(f"Saved: {PATH_OUTPUT}")
print(f"Saved: {PATH_OUTPUT_UNICOS}")

## 12. Line charts: unique user reports by date and diagnosis

- **Primary diagnosis:** Group by report date (`report_created_at` normalized) and `primary_diagnosis`. For each (date, primary_diagnosis) compute the number of **unique user_id**. Rank diseases by total number of reports (sum of unique users across days). Plot one line per disease (top N by total reports).
- **Secondary diagnosis:** Same logic for `secondary_diagnoses` (split by `"; "` so each secondary disease is counted). Group by date and secondary diagnosis, count unique user_id, rank by total reports, plot top N lines.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

COL_DATE = "report_created_at"
COL_USER = "user_id"
TOP_N = 15

df_plot = df_classificado.copy()
df_plot["date"] = pd.to_datetime(df_plot[COL_DATE], errors="coerce").dt.normalize()
df_plot = df_plot[df_plot["date"].notna()]

primary_agg = (
    df_plot.groupby(["date", "primary_diagnosis"])[COL_USER]
    .nunique()
    .reset_index()
    .rename(columns={COL_USER: "n_unique_users", "primary_diagnosis": "diagnosis"})
)
rank_primary = (
    primary_agg.groupby("diagnosis")["n_unique_users"]
    .sum()
    .sort_values(ascending=False)
    .head(TOP_N)
    .index.tolist()
)
primary_pivot = primary_agg[primary_agg["diagnosis"].isin(rank_primary)].pivot(
    index="date", columns="diagnosis", values="n_unique_users"
).fillna(0)
primary_pivot = primary_pivot.reindex(columns=rank_primary)

fig1, ax1 = plt.subplots(figsize=(12, 5))
for col in primary_pivot.columns:
    ax1.plot(primary_pivot.index, primary_pivot[col], label=col, alpha=0.8)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax1.set_xlabel("Report date")
ax1.set_ylabel("Unique user IDs")
ax1.set_title("Primary diagnosis: unique user reports per day (top {} diseases by total reports)".format(TOP_N))
ax1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

sec_rows = []
for _, row in df_plot[["date", COL_USER, "secondary_diagnoses"]].drop_duplicates().iterrows():
    if pd.isna(row["secondary_diagnoses"]) or str(row["secondary_diagnoses"]).strip() == "":
        continue
    for d in str(row["secondary_diagnoses"]).split("; "):
        d = d.strip()
        if d:
            sec_rows.append({"date": row["date"], COL_USER: row[COL_USER], "diagnosis": d})
df_sec = pd.DataFrame(sec_rows)
if len(df_sec) > 0:
    secondary_agg = (
        df_sec.groupby(["date", "diagnosis"])[COL_USER]
        .nunique()
        .reset_index()
        .rename(columns={COL_USER: "n_unique_users"})
    )
    rank_secondary = (
        secondary_agg.groupby("diagnosis")["n_unique_users"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_N)
        .index.tolist()
    )
    secondary_pivot = secondary_agg[secondary_agg["diagnosis"].isin(rank_secondary)].pivot(
        index="date", columns="diagnosis", values="n_unique_users"
    ).fillna(0)
    secondary_pivot = secondary_pivot.reindex(columns=rank_secondary)

    fig2, ax2 = plt.subplots(figsize=(12, 5))
    for col in secondary_pivot.columns:
        ax2.plot(secondary_pivot.index, secondary_pivot[col], label=col, alpha=0.8)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=45)
    ax2.set_xlabel("Report date")
    ax2.set_ylabel("Unique user IDs")
    ax2.set_title("Secondary diagnosis: unique user reports per day (top {} diseases by total reports)".format(TOP_N))
    ax2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No secondary diagnoses to plot.")